# BioOperatorEnv — GRPO Training (Colab)

Trains a Qwen Instruct model with GRPO via TRL + Unsloth + LoRA against `BioOperatorEnv`.

**Recommended runtime:** Colab T4 / L4 / A100 (GPU enabled).

## How to use this notebook

- **In Colab (you start with a fresh runtime, no repo):** run **all** cells from the top. The first two cells clone the repo and install Unsloth + TRL.
- **Locally (if you somehow have a CUDA box):** **skip the first two cells** ("Colab setup"). Start at the path-setup cell.

## (Colab only) 1. Install + clone

Edit the `git clone` line below to point at YOUR fork of this repo.

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth==2024.10 trl peft datasets bitsandbytes accelerate wandb
!pip install -q numpy scipy pandas pydantic matplotlib tqdm fastapi uvicorn

In [ ]:
import os
REPO_URL = 'https://github.com/Json604/openenv-bioreactor.git'
REPO_DIR = 'openenv-bioreactor'
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR

## 2. Path setup (works in Colab and locally)

In [ ]:
import os, sys
from pathlib import Path
candidates = [Path.cwd(), Path.cwd() / 'openenv-bioreactor', Path.cwd() / 'meta_env', Path.cwd().parent]
repo_root = next((p for p in candidates if (p / 'bioperator_env').exists()), None)
assert repo_root is not None, f'cannot locate bioperator_env (cwd={Path.cwd()})'
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))
print('repo root:', repo_root)

## 3. Smoke test the env loads

In [ ]:
from bioperator_env.env import BioOperatorEnv
env = BioOperatorEnv(task_id='do-recovery-easy', seed=42)
obs = env.reset()
print('time_h:', obs.time_h)
print('alarm:', obs.alarm)
print('DO%:', obs.measurements['dissolved_oxygen_pct'])

## 4. Build GRPO dataset (256 prompts from easy DO recovery)

In [ ]:
from training.rollout import build_dataset
from datasets import Dataset
rows = build_dataset(num_samples=256, task_ids=['do-recovery-easy'], seed=0)
ds = Dataset.from_list(rows)
print(f'Dataset rows: {len(ds)}')
print('First prompt tail:', rows[0]['prompt'][-200:])

## 5. Load Qwen 2.5 3B Instruct + LoRA via Unsloth

In [ ]:
from unsloth import FastLanguageModel
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, max_seq_length=2048, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.0, bias='none', use_gradient_checkpointing='unsloth', random_state=42,
)

## 6. Capture untrained-Qwen baseline (the 'before' picture)

In [ ]:
import json, re, os
from bioperator_env.prompt import build_prompt

def run_episode(env, model, tokenizer, max_new_tokens=64):
    obs = env.reset()
    do, rewards, sample_actions = [obs.measurements['dissolved_oxygen_pct']], [], []
    done = False
    while not done:
        prompt = build_prompt(obs)
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.5,
                              do_sample=True, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        m = re.search(r'\{[^{}]*\}', text, re.DOTALL)
        try:
            a = json.loads(m.group(0))
        except Exception:
            a = {'feed_delta_L_h': 0, 'aeration_delta_vvm': 0.0, 'agitation_delta_rpm': 0}
        obs, r, done, info = env.step(a)
        do.append(obs.measurements['dissolved_oxygen_pct'])
        rewards.append(r)
        if env.step_count <= 3:
            sample_actions.append((env.step_count, a))
    return dict(do=do, rewards=rewards, cum=sum(rewards), violations=env.safety_violations,
                sample_actions=sample_actions)

env = BioOperatorEnv(task_id='do-recovery-medium', seed=999)
untrained = run_episode(env, model, tokenizer)
print('UNTRAINED  cum_reward =', untrained['cum'], '  safety_violations =', untrained['violations'])
for step, a in untrained['sample_actions']:
    print(f'  step {step} action: {a}')

## 7. Train (the actual GRPO loop)

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from training.reward_fn import reward_fn

cfg = GRPOConfig(
    output_dir='checkpoints/qwen3-bioperator-lora',
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_completion_length=128,
    max_steps=200,
    beta=0.04,
    temperature=0.7,
    logging_steps=5,
    save_steps=50,
    report_to='none',
)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=cfg,
    train_dataset=ds,
)
trainer.train()

## 8. Save adapter and plot reward curve

In [ ]:
os.makedirs('checkpoints/qwen3-bioperator-lora', exist_ok=True)
model.save_pretrained('checkpoints/qwen3-bioperator-lora')
tokenizer.save_pretrained('checkpoints/qwen3-bioperator-lora')
print('Saved adapter to checkpoints/qwen3-bioperator-lora')

In [ ]:
import matplotlib.pyplot as plt
log_history = trainer.state.log_history
rewards = [(h['step'], h.get('reward')) for h in log_history if 'reward' in h]
if rewards:
    xs, ys = zip(*rewards)
    plt.figure(figsize=(8,4))
    plt.plot(xs, ys, marker='o')
    plt.xlabel('GRPO step'); plt.ylabel('Mean reward')
    plt.title('BioOperatorEnv GRPO training reward')
    plt.grid(alpha=0.3)
    os.makedirs('results', exist_ok=True)
    plt.savefig('results/reward_curve.png', dpi=120, bbox_inches='tight')
    plt.show()

## 9. Capture trained-Qwen on the SAME seed (the 'after' picture)

In [ ]:
env = BioOperatorEnv(task_id='do-recovery-medium', seed=999)
trained = run_episode(env, model, tokenizer)
print('TRAINED    cum_reward =', trained['cum'], '  safety_violations =', trained['violations'])
for step, a in trained['sample_actions']:
    print(f'  step {step} action: {a}')

## 10. Side-by-side comparison plot

In [ ]:
import numpy as np
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(untrained['do'], label='untrained Qwen', linewidth=2, color='#e34a33')
axes[0].plot(trained['do'], label='trained Qwen (GRPO+LoRA)', linewidth=2, color='#31a354')
axes[0].axhline(20, color='grey', linestyle='--', label='DO_min_safe=20%')
axes[0].set_ylabel('Dissolved O2 %')
axes[0].set_title('Before / After GRPO training (do-recovery-medium, seed=999)')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(np.cumsum(untrained['rewards']), label='untrained cum reward', color='#e34a33')
axes[1].plot(np.cumsum(trained['rewards']), label='trained cum reward', color='#31a354')
axes[1].set_xlabel('Episode step'); axes[1].set_ylabel('Cumulative reward')
axes[1].legend(); axes[1].grid(alpha=0.3)
fig.tight_layout()
fig.savefig('results/before_after_demo.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nUNTRAINED -> TRAINED:')
print(f'  cumulative reward:    {untrained["cum"]:.2f}  ->  {trained["cum"]:.2f}')
print(f'  safety violations:    {untrained["violations"]}  ->  {trained["violations"]}')
print(f'  min DO reached:       {min(untrained["do"]):.1f}%  ->  {min(trained["do"]):.1f}%')